In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Memuat dataset
data = pd.read_csv('loan_data.csv')

# Menampilkan data awal
print("Data Awal:")
data.head()

Data Awal:


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [ ]:
# Langkah 1: Pengkodean fitur kategorikal
encoder_label = {}
kolom_kategorikal = ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
for kolom in kolom_kategorikal:
    encoder = LabelEncoder()
    data[kolom] = encoder.fit_transform(data[kolom])
    encoder_label[kolom] = encoder

print("Data Setelah Pengkodean:")
data.head()

Data Setelah Pengkodean:


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,0,4,71948.0,0,3,35000.0,4,16.02,0.49,3.0,561,0,1
1,21.0,0,3,12282.0,0,2,1000.0,1,11.14,0.08,2.0,504,1,0
2,25.0,0,3,12438.0,3,0,5500.0,3,12.87,0.44,3.0,635,0,1
3,23.0,0,1,79753.0,0,3,35000.0,3,15.23,0.44,2.0,675,0,1
4,24.0,1,4,66135.0,1,3,35000.0,3,14.27,0.53,4.0,586,0,1


In [ ]:
# Langkah 2: Normalisasi
normalisasi = MinMaxScaler()
kolom_normalisasi = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']
data[kolom_normalisasi] = normalisasi.fit_transform(data[kolom_normalisasi])

print("Data Setelah Normalisasi:")
data.head()

Data Setelah Normalisasi:


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,0.016129,0,4,0.008891,0.000,3,1.000000,4,0.727023,0.742424,0.035714,0.371739,0,1
1,0.008065,0,3,0.000595,0.000,2,0.014493,1,0.392318,0.121212,0.000000,0.247826,1,0
2,0.040323,0,3,0.000617,0.024,0,0.144928,3,0.510974,0.666667,0.035714,0.532609,0,1
3,0.024194,0,1,0.009976,0.000,3,1.000000,3,0.672840,0.666667,0.000000,0.619565,0,1
4,0.032258,1,4,0.008082,0.008,3,1.000000,3,0.606996,0.803030,0.071429,0.426087,0,1


In [ ]:
#simpan hasil ke file baru
normalisasi_file ='normalisasi_dataset.csv'
data.to_csv(normalisasi_file, index =False)
print(f"\nData yang sudah dinormalisasi disimpan ke: {normalisasi_file}")


Data yang sudah dinormalisasi disimpan ke: normalisasi_dataset.csv


In [ ]:
# Langkah 3: Seleksi Fitur
X = data.drop(columns=['loan_status'])
y = data['loan_status']
selektor = SelectKBest(score_func=f_classif, k='all')
X_baru = selektor.fit_transform(X, y)

# Menampilkan skor fitur
skor_fitur = pd.DataFrame({
    'Fitur': X.columns,
    'Skor': selektor.scores_
}).sort_values(by='Skor', ascending=False)

print("Skor Seleksi Fitur:")
print(skor_fitur)

# Membagi data menjadi data latih dan uji
X_latih, X_uji, y_latih, y_uji = train_test_split(X_baru, y, test_size=0.2, random_state=42)

Skor Seleksi Fitur:
                             Fitur          Skor
12  previous_loan_defaults_on_file  18824.727466
9              loan_percent_income   7824.794030
8                    loan_int_rate   5574.454260
5            person_home_ownership   2602.906481
3                    person_income    845.525887
6                        loan_amnt    528.213632
7                      loan_intent    235.522228
0                       person_age     20.763596
4                   person_emp_exp     18.883771
10      cb_person_cred_hist_length      9.926174
11                    credit_score      2.631606
2                 person_education      0.137280
1                    person_gender      0.014178


In [ ]:
# Langkah 4: Regresi Logistik
model = LogisticRegression()
model.fit(X_latih, y_latih)

# Prediksi
y_prediksi = model.predict(X_uji)

# Evaluasi
print("Laporan Klasifikasi:")
print(classification_report(y_uji, y_prediksi))

print("Akurasi Model Regresi Logistik:", accuracy_score(y_uji, y_prediksi))

def tampilkan_hasil_akhir():
    print("Data Akhir dengan Fitur Terseleksi:")
    print(pd.DataFrame(X_baru, columns=X.columns))

if __name__ == "__main__":
    tampilkan_hasil_akhir()


Laporan Klasifikasi:
              precision    recall  f1-score   support

           0       0.92      0.94      0.93      6990
           1       0.77      0.73      0.75      2010

    accuracy                           0.89      9000
   macro avg       0.84      0.83      0.84      9000
weighted avg       0.89      0.89      0.89      9000

Akurasi Model Regresi Logistik: 0.8893333333333333
Data Akhir dengan Fitur Terseleksi:
       person_age  person_gender  person_education  person_income  \
0        0.016129            0.0               4.0       0.008891   
1        0.008065            0.0               3.0       0.000595   
2        0.040323            0.0               3.0       0.000617   
3        0.024194            0.0               1.0       0.009976   
4        0.032258            1.0               4.0       0.008082   
...           ...            ...               ...            ...   
44995    0.056452            1.0               0.0       0.005557   
44996    0.13